# Stage 2 - Data Cleaning

## 01_cleaning.ipynb

#### Prepares the raw South African crime statistics dataset for Exploratory Data Analysis (EDA) by:   

- Loading the `sa_crime_raw` CSV from `data/raw/sa_crime_raw.csv`,       
- Inspecting and verifying data quality findings from Stage 1 programmatically,     
- Renaming columns to match the project schema,
- Mapping the province abbreviation entries to their full names,
- Renaming the crime category entries to shorter, more readable, chart-friendly versions,         
- Parsing the `Financial Year` field from a text string to an integer of 4 characters,         
- Exporting the clean `sa_crime_clean` CSV to `data/processed/sa_crime_clean.csv` for subsequent stages in the data pipeline, and    
- Establishing and exporting a relational database `sa_crime.db` populated with cleaned dataframe for querying in Stage 7.


#### Data source: [Crime Statistics of South Africa (2011-2023)](https://www.kaggle.com/datasets/harutyunagababyan/crime-stats-of-south-africa-2011-2023)

---------------


Imports necessary libraries: `pandas` for data manipulation; `sqlite3` for database creation; `os` for directory navigation:

In [1]:
import pandas as pd
import sqlite3
import os     

Navigates from notebook's location to project root, i.e. since notebooks live in `notebooks/`, goes up one level to reach project root:

In [2]:
dir_nb = os.getcwd()
project_root = os.path.dirname(dir_nb)   # retrieves project root directory     
os.chdir(project_root)                   # changes directory to project root

print(os.getcwd())                       # prints project root directory

C:\Users\Wits-User\Desktop\PROJECTS\sa-crime-statistics


Loads raw data from file:

In [3]:
crime_df = pd.read_csv('data/raw/sa_crime_raw.csv')

print('Raw crime statistics dataset read into dataframe!')

Raw crime statistics dataset read into dataframe!


Programmatically confirms observations made in Stage 1 during profiling phase:

In [4]:
print('Shape:',                   crime_df.shape)                     # expected: (756,4)             
print('Columns:',                 crime_df.columns.tolist())          
print('Data Types:\n',            crime_df.dtypes, sep = '')          #'sep' argument for alignment purposes
print('Null entries:\n',          crime_df.isnull().sum(), sep = '')  # expected: all 0
print('Duplicates:',              crime_df.duplicated().sum())        # expected: 0
print('Zero entries:\n',         (crime_df == 0).sum(), sep = '')     # expected: all 0
print('Negative count entries:', (crime_df['Count'] < 0).sum())       # expected: 0

Shape: (756, 4)
Columns: ['Geography', 'Crime Category', 'Financial Year', 'Count']
Data Types:
Geography           str
Crime Category      str
Financial Year      str
Count             int64
dtype: object
Null entries:
Geography         0
Crime Category    0
Financial Year    0
Count             0
dtype: int64
Duplicates: 0
Zero entries:
Geography         0
Crime Category    0
Financial Year    0
Count             0
dtype: int64
Negative count entries: 0


Renames column labels to match project schema. This step is executed without capital standardisation, as all column names were confirmed as consistent during Stage 1:

In [5]:
column_labels = crime_df.columns                # extracts column names and index positions (column order confirmed above)

crime_df.rename(columns = {
          column_labels[0] : 'province',        # uses index positions in place of column names as strings to avoid typos
          column_labels[1] : 'crime_category',
          column_labels[2] : 'financial_year',
          column_labels[3] : 'incident_count',
}, inplace = True)

print(crime_df.columns.tolist())                # expected: ['province', 'crime_category', 'financial_year', 'incident_count']   

['province', 'crime_category', 'financial_year', 'incident_count']


Province names in the raw dataset (`province` column) are stored as abbreviations. To ensure proper readability across all outputs in subsequent workflow stages, this step maps the province abbreviations to their full names. Furthermore, full province names ensure direct compatibility with the shapefile province name column in the geospatial notebook:

In [6]:
map_province = {'EC'  :   'Eastern Cape',                 # creates mapping dictionary with full province names
                'FS'  :   'Free State',
                'GT'  :   'Gauteng',
                'KZN' :   'KwaZulu-Natal',
                'LIM' :   'Limpopo',
                'MP'  :   'Mpumalanga',
                'NC'  :   'Northern Cape',
                'NW'  :   'North West',
                'WC'  :   'Western Cape'}

crime_df['province'] = crime_df['province'].map(map_province)       # replaces abbreviations with full names   

print(crime_df['province'].isnull().sum())    # verifies if all names are mapped - expected: 0

print(crime_df['province'].unique())          # verifies province names to ensure no typos made
print(crime_df['province'].nunique())         # verifies number of unique elements - expected: 9

0
<ArrowStringArray>
[ 'Eastern Cape',    'Free State',       'Gauteng', 'KwaZulu-Natal',
       'Limpopo',    'Mpumalanga', 'Northern Cape',    'North West',
  'Western Cape']
Length: 9, dtype: str
9


Several crime category entries in the raw dataset (`crime_category` column) contains the suffix `Crimes`. To remove redundancy and ensure proper readability across all outputs in subsequent workflow stages, several of the category labels are renamed to shorter, more chart-friendly versions:

In [7]:
map_category = {'Contact Crimes'                                :  'Contact',                            
                'Property Related Crimes'                       :  'Property Related',
                'Other Serious Crimes'                          :  'Other Serious',
                'Crimes Detected as a Result of Police Action'  :  'Police Action',
                'Contact Related Crimes'                        :  'Contact Related',
                'Aggravated Robberies'                          :  'Aggravated Robberies',
                'Sexual Offences'                               :  'Sexual Offences'}

crime_df['crime_category'] = crime_df['crime_category'].map(map_category)    # replaces category names with revisions  

print(crime_df['crime_category'].isnull().sum())    # verifies if all names are mapped - expected: 0

print(crime_df['crime_category'].unique())          # verifies province names to ensure no typos made
print(crime_df['crime_category'].nunique())         # verifies number of unique elements - expected: 7

0
<ArrowStringArray>
['Aggravated Robberies',              'Contact',      'Contact Related',
        'Police Action',        'Other Serious',     'Property Related',
      'Sexual Offences']
Length: 7, dtype: str
7


The elements situated in the `financial_year` column are stored as `20XX/20XX` text strings. This step parses the year entries to the first 4 characters of the text strings - yields the start year as an integer, which complies with conventional SAPS publications:

In [8]:
crime_df['financial_year'] = crime_df['financial_year'].astype(str).str[:4].astype(int)   # extracts and parses text string to four characters

print(crime_df['financial_year'].unique())    # displays unique elements - expected: [2011, 2012, 2013, ..., 2022] 

print(crime_df['financial_year'].dtype)       # displays data type - expected: int64       

[2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022]
int64


Exports the processed dataframe `crime_df` to file - feeds tools (SQL, R, MATLAB, Power BI) used in subsequent stages of the data pipeline:

In [9]:
crime_df.to_csv('data/processed/sa_crime_clean.csv', index = False)

print('Export complete. Shape:', crime_df.shape)         # confirms shape of exported data - expected: (756,4)

Export complete. Shape: (756, 4)


Creates `sa_crime` relational database and populates it with the clean `sa_crime` dataset:

In [10]:
os.makedirs('data/sql', exist_ok = True)     # creates a directory if it does not exist

conn = sqlite3.connect('data/sql/sa_crime.db')     # establishes connection to SQLite and creates 'sa_crime' database

crime_df.to_sql('sa_crime',conn, if_exists = 'replace', index = False)    # 'if_exists' argument ensures recreation of table each execution

row_verification = pd.read_sql('SELECT COUNT(*) AS row_count FROM sa_crime', conn)     # verifies row count using query
print(row_verification)                                                                # expected: 756

conn.close()     # closes connection

   row_count
0        756


The outputs of this notebook are as follows: `sa_crime_clean.csv` (756 rows, 4 columns) and `sa_crime.db` (database populated with `sa_crime_clean` data). Both outputs are ready for use in subsequent pipeline stages. 